In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
from langchain_community.tools.tavily_search import TavilySearchResults

In [10]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from typing import TypedDict, List, Dict, Optional, Literal
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI

class AgentState(TypedDict):
    current_step: str
    startup_list: List[Dict]
    selected_startup: Optional[Dict]
    startup_info: Optional[Dict]
    tech_info: Optional[Dict]
    investment_decision: Optional[Literal["투자추천", "투자보류"]]
    report: Optional[str]
    all_investment_decisions: Dict[str, Literal["투자추천", "투자보류"]]
    processed_startups_count: int
    total_startups_count: int
    messages: List[Dict]

In [5]:
def load_pdf_text(file_path: str) -> str:
    loader = PyMuPDFLoader(file_path)
    pages = loader.load()
    return "\n".join([p.page_content for p in pages])

def extract_structured_info(text: str) -> str:
    with open("prompts/startup_info_prompt.txt", "r", encoding="utf-8") as f:
        template = f.read()
    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": text})

def split_to_documents(extracted_text: str, pdf_name: str) -> list[Document]:
    sections = extracted_text.split("\n---")[0].split("\n")
    documents = []
    current_section = ""
    buffer = []

    for line in sections:
        if not line.strip():
            continue
        if ":" in line and not line.startswith("  "):  # New section
            if buffer:
                documents.append(
                    Document(page_content="\n".join(buffer), metadata={"type": current_section, "source": pdf_name})
                )
            current_section, value = line.split(":", 1)
            buffer = [f"{current_section.strip()}: {value.strip()}"]
        else:
            buffer.append(line.strip())

    if buffer:
        documents.append(
            Document(page_content="\n".join(buffer), metadata={"type": current_section, "source": pdf_name})
        )
    return documents

def store_in_chroma(documents: list[Document], collection_name: str, persist_dir: str = "chroma_store"):
    os.makedirs(persist_dir, exist_ok=True)
    chroma = Chroma.from_documents(
        documents,
        embedding=OpenAIEmbeddings(),
        persist_directory=persist_dir,
        collection_name=collection_name
    )
    chroma.persist()

def save_structured_text_to_file(startup_name: str, structured_text: str, web_news_docs: list[Document] = None, output_dir="data/outputs") -> str:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{startup_name}_info.txt")
    
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(structured_text)
        f.write("\n\n" + "="*50 + "\n")
        f.write("웹서치 뉴스 요약 결과\n")
        f.write("="*50 + "\n\n")
        
        if web_news_docs:
            for i, doc in enumerate(web_news_docs, 1):
                f.write(f"{i}. {doc.page_content.strip()}\n\n")
        else:
            f.write("웹서치 뉴스 없음\n")
    
    return output_path

def get_web_news_fallback(query: str, k: int = 2) -> list[Document]:
    search = TavilySearchResults(k=k)
    results = search.invoke(query)
    return [
        Document(
            page_content=f"{item.get('content')}\n\n링크: {item.get('url')}",
            metadata={"type": "웹서치", "source": query}
        )
        for item in results
    ]

if __name__ == "__main__":

    startup_name = "seedn"
    pdf_path = f"data/{startup_name}.pdf"

    # 1. PDF 로딩
    raw_text = load_pdf_text(pdf_path)

    # 2. LLM 요약
    structured_text = extract_structured_info(raw_text)

    # 3. 웹서치 뉴스
    web_news_docs = get_web_news_fallback(f"{startup_name} 스타트업 뉴스")

    # 4. 텍스트 저장 (웹서치 내용 포함)
    output_txt_path = save_structured_text_to_file(startup_name, structured_text, web_news_docs)
    print(f"정제된 텍스트 + 뉴스 파일 저장 완료: {output_txt_path}")

    # 5. PDF 문서 분할 + 저장
    documents = split_to_documents(structured_text, pdf_name=pdf_path)
    store_in_chroma(documents, collection_name=startup_name)

    # 6. 웹서치 뉴스 저장
    store_in_chroma(web_news_docs, collection_name=startup_name)

정제된 텍스트 + 뉴스 파일 저장 완료: data/outputs/seedn_info.txt


/var/folders/7h/cfwt31ps3wl6r0thkyyt156w0000gn/T/ipykernel_8238/3863724569.py:48: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma.persist()


In [4]:
# PDF 파일을 텍스트로 로드하는 함수
def load_pdf_text(file_path: str) -> str:                   
    loader = PyMuPDFLoader(file_path)                       # PDF 로더 초기화
    pages = loader.load()                                   # 모든 페이지 로드
    return "\n".join([p.page_content for p in pages])       # 페이지 내용을 하나의 문자열로 결합
    
# LLM을 사용해 구조화된 정보를 추출하는 함수
def extract_structured_info(text: str) -> str:
    with open("prompts/startup_info_prompt.txt", "r", encoding="utf-8") as f:
        template = f.read()

    llm = ChatOpenAI(model="gpt-4o", temperature=0)         # LLM 모델 지정
    prompt = PromptTemplate.from_template(template)         # 템플릿을 Prompt 객체로 변환
    chain = prompt | llm | StrOutputParser()                # 프롬프트 → LLM → 출력 파서 체인 구성
    return chain.invoke({"text": text})                     # LLM에 텍스트 전달하고 결과 반환

# 구조화된 텍스트를 Document 리스트로 변환 (Chroma 저장용)
def split_to_documents(extracted_text: str, pdf_name: str, company_name: str) -> list[Document]:
    sections = extracted_text.split("\n---")[0].split("\n") # "---" 구분 전까지 섹션 추출
    documents = []
    current_section = ""
    buffer = []

    for line in sections:
        if not line.strip():
            continue
        if ":" in line and not line.startswith("  "):       # 새로운 섹션의 시작
            if buffer:
                documents.append(
                    Document(
                        page_content="\n".join(buffer),
                        metadata={"type": current_section, "source": pdf_name, "agent_type": "startup_agent", "company_name": company_name}
                    )
                )
            current_section, value = line.split(":", 1)
            buffer = [f"{current_section.strip()}: {value.strip()}"]
        else:
            buffer.append(line.strip())
    # 마지막 버퍼 내용 저장
    if buffer:
        documents.append(
            Document(
                page_content="\n".join(buffer),
                metadata={"type": current_section, "source": pdf_name, "agent_type": "startup_agent", "company_name": company_name}
            )
        )
    return documents

# ChromaDB에 Document 저장 (collection_name에 저장)
def store_in_chroma(documents: list[Document], collection_name: str, persist_dir: str = "chroma_store"):
    os.makedirs(persist_dir, exist_ok=True)
    
    # 문서별 고유 ID 생성 (회사명, 문서타입, 인덱스를 조합)
    ids = []
    for i, doc in enumerate(documents):
        company = doc.metadata.get("company_name") or "unknown"
        doc_type = doc.metadata.get("type") or f"doc{i}"
        ids.append(f"{company}_{doc_type}_{i}")

    # 벡터 저장소에 문서 저장
    chroma = Chroma.from_documents(
        documents=documents,
        embedding=OpenAIEmbeddings(),
        persist_directory=persist_dir,
        collection_name=collection_name,
        ids=ids
    )
    chroma.persist()

# 구조화된 텍스트 + 뉴스 결과를 TXT 파일로 저장
def save_structured_text_to_file(startup_name: str, structured_text: str, web_news_docs: list[Document] = None, output_dir="data/outputs") -> str:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{startup_name}_info.txt")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(structured_text)
        f.write("\n\n" + "="*50 + "\n")
        f.write("웹서치 뉴스 요약 결과\n")
        f.write("="*50 + "\n\n")

        if web_news_docs:
            for i, doc in enumerate(web_news_docs, 1):
                f.write(f"{i}. {doc.page_content.strip()}\n\n")
        else:
            f.write("웹서치 뉴스 없음\n")

    return output_path

# 웹서치 도구를 통해 최신 뉴스 결과를 가져와 Document 형태로 반환
def get_web_news_fallback(query: str, k: int = 2) -> list[Document]:
    search = TavilySearchResults(k=k)
    results = search.invoke(query)
    return [
        Document(
            page_content=f"{item.get('content')}\n\n링크: {item.get('url')}",
            metadata={"type": "웹서치", "source": query, "agent_type": "startup_agent", "company_name": query.split()[0]}
        )
        
        
        for item in results
    ]

In [5]:
if __name__ == "__main__":
    startup_name = "seedn"
    pdf_path = f"data/{startup_name}.pdf"

    # 1. PDF 로딩
    raw_text = load_pdf_text(pdf_path)

    # 2. LLM 요약
    structured_text = extract_structured_info(raw_text)

    # 3. 웹서치 뉴스
    web_news_docs = get_web_news_fallback(f"{startup_name} 스타트업 뉴스")

    # 4. 텍스트 저장 (웹서치 내용 포함)
    output_txt_path = save_structured_text_to_file(startup_name, structured_text, web_news_docs)
    print(f"정제된 텍스트 + 뉴스 파일 저장 완료: {output_txt_path}")

    # 5. PDF 문서 분할 + 저장
    documents = split_to_documents(structured_text, pdf_name=pdf_path, company_name=startup_name)
    store_in_chroma(documents, collection_name="company_data")

    # 6. 웹서치 뉴스 저장
    store_in_chroma(web_news_docs, collection_name="company_data")

KeyboardInterrupt: 

In [6]:
def collect_startup_info(state: AgentState) -> AgentState:
    startup = state["selected_startup"]
    if not startup:
        raise ValueError("선택된 스타트업 정보가 없습니다.")

    startup_name = startup["name"]
    pdf_path = f"data/{startup_name}.pdf"

    if not os.path.exists(pdf_path):
        return {
            **state,
            "startup_info": {"status": "0", "reason": f"{pdf_path} 없음"},
            "messages": state["messages"] + [{"role": "system", "content": f"{startup_name} PDF 없음"}],
        }

    # 1. PDF + 요약
    raw_text = load_pdf_text(pdf_path)
    structured_text = extract_structured_info(raw_text)

    # 2. 웹 뉴스
    web_news_docs = get_web_news_fallback(f"{startup_name} 스타트업 뉴스")

    # 3. 저장
    text_path = save_structured_text_to_file(startup_name, structured_text, web_news_docs)
    documents = split_to_documents(structured_text, pdf_name=pdf_path, company_name=startup_name)
    store_in_chroma(documents, collection_name="company_data")
    store_in_chroma(web_news_docs, collection_name="company_data")

    return {
        **state,
        "startup_info": {
            "status": "1",
            "company": startup_name,
            "pdf": pdf_path,
            "text_path": text_path
        },
        "messages": state["messages"] + [{"role": "system", "content": f"{startup_name} 정보 수집 완료"}],
    }

In [7]:
if __name__ == "__main__":
    # 테스트용 입력 상태 정의
    test_state = {
        "current_step": "스타트업_정보_수집",
        "startup_list": [{"name": "seedn"}],
        "selected_startup": {"name": "seedn"},
        "messages": [],
        "startup_info": None,
        "tech_info": None,
        "investment_decision": None,
        "report": None,
        "all_investment_decisions": {},
        "processed_startups_count": 0,
        "total_startups_count": 1,
    }

    # 함수 실행
    new_state = collect_startup_info(test_state)

    # 결과 출력
    from pprint import pprint
    pprint(new_state["startup_info"])
    print("메시지 로그:")
    for msg in new_state["messages"]:
        print(f"{msg['role']}: {msg['content']}")

/var/folders/7h/cfwt31ps3wl6r0thkyyt156w0000gn/T/ipykernel_8492/1635895656.py:68: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma.persist()


{'company': 'seedn',
 'pdf': 'data/seedn.pdf',
 'status': '1',
 'text_path': 'data/outputs/seedn_info.txt'}
메시지 로그:
system: seedn 정보 수집 완료


In [8]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [18]:
from langchain_community.tools.tavily_search.tool import TavilySearchResults
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document
from bs4 import BeautifulSoup  # ✅ 추가
import os
import uuid
from typing import TypedDict, List, Dict, Optional, Literal

class AgentState(TypedDict):
    current_step: str
    startup_list: List[Dict]
    selected_startup: Optional[Dict]
    startup_info: Optional[Dict]
    tech_info: Optional[Dict]
    investment_decision: Optional[Literal["투자추천", "투자보류"]]
    report: Optional[str]
    all_investment_decisions: Dict[str, Literal["투자추천", "투자보류"]]
    processed_startups_count: int
    total_startups_count: int
    messages: List[Dict]

def clean_html_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator="\n")

    # ✅ 불필요한 패턴 필터링
    filters = [
        "무단 전재 및 재배포 금지",
        "저작권자",
        "관련기사",
        "SNS 공유하기",
        "광고 문의",
        "네이버 홈",
        "All rights reserved", "무단복제", 
        "기자의 다른 기사 보기", "기사제보", "페이스북", "트위터", "카카오스토리", 
        "네이버 뉴스", "본문 시작", "기사 본문", "닫기"
    ]
    for pattern in filters:
        text = text.replace(pattern, "")
    
    # ✅ 공백 정리
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

def tech_exploration(state: AgentState, save_txt: bool = True) -> AgentState:
    company = state["selected_startup"]["name"]
    search = TavilySearchResults(k=5)
    results = search.run(f"{company} 기술력 OR 핵심 기술 OR AI OR IoT OR 특허")
    urls = [r["url"] for r in results]

    loader = WebBaseLoader(urls)
    docs = loader.load()

    # ✅ HTML 정제 적용
    texts = [clean_html_text(doc.page_content) for doc in docs]

    llm = ChatOpenAI(temperature=0.2)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    summaries = []
    for text in texts:
        msg = llm.invoke(f"{text}\n\n이 기업의 기술을 요약해줘.")
        summaries.append(msg.content)

    summary_collection_name = f"{uuid.uuid4().hex[:8]}_tech_summary"
    persist_dir = "../chroma_store"
    summary_docs = [
        Document(page_content=s, metadata={
            "agent_type": "tech_summary",
            "company_name": company
        }) for s in summaries
    ]
    summary_vectorstore = Chroma.from_documents(
        documents=summary_docs,
        embedding=embeddings,
        collection_name=summary_collection_name,
        persist_directory=persist_dir
    )

    final_response = llm.invoke("".join(summaries) + "\n\n이 스타트업의 기술적 경쟁력은 무엇인가요?")
    final_summary = final_response.content
    final_doc = Document(
        page_content=final_summary,
        metadata={"agent_type": "tech_agent", "company_name": company}
    )
    summary_vectorstore.add_documents([final_doc])

    if save_txt:
        os.makedirs("../text_backup", exist_ok=True)
        vector_txt_path = f"../text_backup/{company}_vector_content.txt"
        all_docs = summary_vectorstore.similarity_search("기술", k=100)
        with open(vector_txt_path, "w", encoding="utf-8") as f:
            for i, d in enumerate(all_docs, 1):
                f.write(f"[{i}] {d.metadata.get('agent_type')} | {d.metadata.get('company_name')}\n{d.page_content}\n\n")
        print(f"✅ 벡터 DB 저장 내용 백업 완료: {vector_txt_path}")

    state["tech_info"] = {
        "summary": final_summary,
        "source_urls": urls
    }
    return state

# 🧪 테스트 실행
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()

    company_name = "프리윌린"
    test_state = {
        "current_step": "기술_탐색_요약",
        "startup_list": [],
        "selected_startup": {"name": company_name},
        "startup_info": None,
        "tech_info": None,
        "investment_decision": None,
        "report": None,
        "all_investment_decisions": {},
        "processed_startups_count": 0,
        "total_startups_count": 10,
        "messages": []
    }

    result_state = tech_exploration(test_state, save_txt=True)

    print("\n📌 최종 기술 요약 응답:", result_state["tech_info"]["summary"])
    print("\n🔗 수집된 기사 URL:")
    for url in result_state["tech_info"]["source_urls"]:
        print(" -", url)

Number of requested results 100 is greater than number of elements in index 6, updating n_results = 6


✅ 벡터 DB 저장 내용 백업 완료: ../text_backup/프리윌린_vector_content.txt

📌 최종 기술 요약 응답: 프리윌린의 기술적 경쟁력은 AI 기술을 활용한 학습 경험의 최적화와 개인화에 있습니다. 회사는 AI를 사용하여 학생들의 학습 스타일을 분석하고 최적의 학습 방법을 추천하는 서비스를 제공하고 있으며, AI를 통해 문제를 분리하고 교육 과정을 최신화하는 작업을 효율적으로 수행하고 있습니다. 또한, AI를 활용한 문제 추천 프로덕트를 개발하고 있어 학습자들에게 더욱 효과적인 학습 경험을 제공하고 있습니다.

또한, 프리윌린은 AI를 활용한 수학 코스웨어를 통해 전 학령기를 아우르는 서비스를 제공하고 있으며, 매쓰플랫을 통해 국내 대표 수학 AI 학습 솔루션으로 자리매김하고 있습니다. 이를 통해 학습의 격차를 줄이고 모든 학습자에게 효과적인 학습 경험을 제공하는 'AI 기반 학습 파트너'로 성장하고자 하는 방향으로 기술적 경쟁력을 강화하고 있습니다.

🔗 수집된 기사 URL:
 - https://www.aitimes.com/news/articleView.html?idxno=162186
 - https://freewheelin-recruit.oopy.io/17dfeddc-e671-8055-8023-fc1019800255
 - https://edu.chosun.com/m/edu_article.html?contid=2025012380071
 - https://www.sedaily.com/NewsView/2GRQ0FKS7K
 - https://www.mk.co.kr/news/it/11305499


In [3]:
from bs4 import BeautifulSoup

def clean_html_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator="\n")
    
company = "씨드앤"
search = TavilySearchResults(k=5)
results = search.run(f"스타트업 {company} 뉴스")
urls = [r["url"] for r in results]

loader = WebBaseLoader(urls)
docs = loader.load()
print(docs)
texts = [clean_html_text(doc.page_content) for doc in docs]

print(texts)
llm = ChatOpenAI(temperature=0.2)

summaries = []
for text in texts:
    msg = llm.invoke(f"{text}\n\n이 기업의 뉴스의 내용이 누락되지 않게 정리해줘.")
    summaries.append(msg.content)


NameError: name 'WebBaseLoader' is not defined

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.tavily_search.tool import TavilySearchResults
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document
from bs4 import BeautifulSoup  # ✅ 추가
import os
import uuid
from typing import TypedDict, List, Dict, Optional, Literal

def clean_html_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator="\n")

    # ✅ 불필요한 패턴 필터링
    filters = [
        "무단 전재 및 재배포 금지",
        "저작권자",
        "관련기사",
        "SNS 공유하기",
        "광고 문의",
        "네이버 홈",
        "All rights reserved", "무단복제", 
        "기자의 다른 기사 보기", "기사제보", "페이스북", "트위터", "카카오스토리", 
        "네이버 뉴스", "본문 시작", "기사 본문", "닫기"
    ]
    for pattern in filters:
        text = text.replace(pattern, "")
    
    # 공백 정리
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

def news_exploration(state: AgentState, save_txt: bool = True) -> AgentState:
    company = state["selected_startup"]["name"]
    search = TavilySearchResults(k=5)
    results = search.run(f"{company} 관련 뉴스 OR 보도자료")
    urls = [r["url"] for r in results]

    loader = WebBaseLoader(urls)
    docs = loader.load()
    print(docs)

    texts = [clean_html_text(doc.page_content) for doc in docs]
    print("여기", texts)

    llm = ChatOpenAI(temperature=0.2)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    summaries = []
    for text in texts:
        msg = llm.invoke(f"{text}\n\n이 뉴스 내용을 핵심 내용이 누락되지 않게 잘 정리해줘.")
        summaries.append(msg.content)

    summary_collection_name = f"{uuid.uuid4().hex[:8]}_news_summary"
    persist_dir = "../chroma_store"
    summary_docs = [
        Document(page_content=s, metadata={
            "agent_type": "news_summary",
            "company_name": company
        }) for s in summaries
    ]
    news_vectorstore = Chroma.from_documents(
        documents=summary_docs,
        embedding=embeddings,
        collection_name=summary_collection_name,
        persist_directory=persist_dir
    )

    final_response = llm.invoke("".join(summaries) + "\n\n이 스타트업에 대한 주요 뉴스 흐름과 시사점은 무엇인가요?")
    final_summary = final_response.content
    final_doc = Document(
        page_content=final_summary,
        metadata={"agent_type": "news_agent", "company_name": company}
    )
    news_vectorstore.add_documents([final_doc])

    if save_txt:
        os.makedirs("../text_backup", exist_ok=True)
        vector_txt_path = f"../text_backup/{company}_news_vector_content.txt"
        all_docs = news_vectorstore.similarity_search("뉴스", k=10)
        with open(vector_txt_path, "w", encoding="utf-8") as f:
            for i, d in enumerate(all_docs, 1):
                f.write(f"[{i}] {d.metadata.get('agent_type')} | {d.metadata.get('company_name')}\n{d.page_content}\n\n")
        print(f"뉴스 벡터 DB 저장 내용 백업 완료: {vector_txt_path}")

    # startup_info에 병합 저장
    if state["startup_info"] is None:
        state["startup_info"] = {}

    state["startup_info"]["news_summary"] = final_summary
    state["startup_info"]["news_source_urls"] = urls

    return state

if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()  # OpenAI API Key 등 환경 변수 로딩

    # 테스트 대상 회사 이름
    company_name = "씨드앤"

    # 테스트용 상태 정의
    test_state: AgentState = {
        "current_step": "뉴스_탐색_요약",
        "startup_list": [],
        "selected_startup": {"name": company_name},
        "startup_info": None,  # ← 여기에 뉴스 요약이 병합 저장됨
        "tech_info": None,
        "investment_decision": None,
        "report": None,
        "all_investment_decisions": {},
        "processed_startups_count": 0,
        "total_startups_count": 1,
        "messages": []
    }

    # 뉴스 탐색 실행
    updated_state = news_exploration(test_state, save_txt=True)

    # 결과 출력
    print("\n뉴스 요약 결과:")
    print(updated_state["startup_info"]["news_summary"])

    print("\n뉴스 기사 출처:")
    for url in updated_state["startup_info"]["news_source_urls"]:
        print(" -", url)